In [ ]:
from sklearn import metrics
import scanpy as sc
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd
import os
import sys
import scib
from load_data import multiHIVE_data, DATASET_SEEDS_PAIRS
from concurrent.futures import ProcessPoolExecutor, as_completed

def _batch_metrics(dataset, seed):
    adata = multiHIVE_data(dataset, seed)

    if adata.obsm.get('latent') is None:
        raise ValueError("Latent representation not found in adata.obsm['latent']")
    if "cell_type" not in adata.obs.columns:
        raise ValueError("Cell type labels not found in adata.obs['cell_type']")
    if "batch" not in adata.obs.columns:
        raise ValueError("Batch labels not found in adata.obs['batch']")
    
    sc.pp.neighbors(adata, use_rep="latent")
    gc_val = scib.me.graph_connectivity(adata, label_key="cell_type")
    asw_batch_val = scib.me.silhouette_batch(adata, batch_key="batch", label_key="cell_type", embed="latent")

    results = {
        "ASW-Batch": [asw_batch_val],
        "Graph Connectivity": [gc_val]
    }
    return results


def compute_clustering_metrics(dataset, seed, re_calculate=False): # parallel wrt resolution
    try:
        if not re_calculate and os.path.exists(f'./Results/{dataset}/{seed}_batch_metrics.csv'):
            print(f"Metrics for {dataset} - {seed} already exist. Skipping computation.")
            return

        os.makedirs(f'./Results/{dataset}/', exist_ok=True)
        
        results = _batch_metrics(dataset, seed)
        
        results = pd.DataFrame(results)
        results.to_csv(f'./Results/{dataset}/{seed}_batch_metrics.csv', index=False)
    except Exception as e:
        print(f"Error processing {dataset} - {seed}: {str(e)}")

In [ ]:
def process_pair(args):
    dataset, seed = args
    compute_clustering_metrics(dataset, seed)
    return dataset, seed

with ProcessPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(process_pair, pair) for pair in DATASET_SEEDS_PAIRS]
    for future in tqdm(as_completed(futures), total=len(DATASET_SEEDS_PAIRS)):
        dataset, seed = future.result()